In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
from transformers import PreTrainedTokenizer

from pivotal_tokens.constants import get_data_dir, get_artifacts_dir
from pivotal_tokens.hf.loading import  load_tokenizer
from pivotal_tokens.repo import SampleRepo, DictRepo


EXP_DIR = get_data_dir() / 'experiments' / 'exp2.1-sps-tokens' / 'exp2.1.2-qwen3-1.7b-sps-tokens-small-prob-threshold'
# EXP_DIR = get_data_dir() / 'experiments' / 'exp2.1-sps-tokens' / 'exp2.1.3-qwen3-1.7b-sps-tokens-small-prob-threshold'
# EXP_DIR = get_data_dir() / 'experiments' / 'exp2.1-sps-tokens' / 'exp2.1.4-qwen3-1.7b-sps-tokens-small-prob-threshold'
REPO_DIR = EXP_DIR / 'data' / 'repo'
PIVOTAL_TOKENS_FILE = EXP_DIR / 'data' / 'pivotal_tokens.csv'

TRACES_FILE = get_artifacts_dir() / 'exp1.1.1-qwen3-1.7b-traces.csv'
PROCESSED_PIVOTAL_TOKENS_FILE = get_artifacts_dir() / 'exp2.1.2-qwen3-1.7b-pivotal-tokens.csv'

QWEN3_1_7B_MODEL_ID = 'Qwen/Qwen3-1.7B'

In [ ]:
tokenizer = load_tokenizer(QWEN3_1_7B_MODEL_ID)

In [ ]:
base_repo = DictRepo(dirpath=REPO_DIR)

In [ ]:
traces_df = pd.read_csv(TRACES_FILE)
traces_df[:2]

In [ ]:
tokens_df = pd.read_csv(PIVOTAL_TOKENS_FILE)
tokens_df['token_ids'] = tokens_df['token_ids'].apply(json.loads)
tokens_df[:2]

In [ ]:
df = pd.merge(traces_df, tokens_df, left_on='id', right_on='sample_id', how='inner')
df[:2]

In [ ]:
def get_init_success_prob(sample_id: str, base_repo: DictRepo) -> float | None:
    repo = SampleRepo(base_repo=base_repo, sample_id=sample_id)
    subdivisions = repo.list(path="subdivisions")

    init_prob = None
    for subdiv in subdivisions:
        data = repo.load(path="subdivisions", key=subdiv)
        if len(data['prefix']) == 0 and \
                data['full_seq'].startswith('<think>') and \
                data['full_seq'].endswith('</think>'):
            init_prob = data['prob_before']
            break
    
    return init_prob

In [ ]:
df['prob_init'] = df['sample_id'].apply(lambda x: get_init_success_prob(x, base_repo))

In [ ]:
df = df[df['is_pivotal'] == True].copy(deep=True)

In [ ]:
len(df)

In [ ]:
df.to_csv(PROCESSED_PIVOTAL_TOKENS_FILE, index=False)